In [1]:
%pip install -q -U ir_measures sentence_transformers

In [1]:
import os

os.environ["HF_TOKEN"] = input()

In [2]:
from datasets import load_dataset, Dataset
import os, json, time, random, pathlib
import requests
import numpy as np
import pandas as pd
import torch
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CrossEncoderRerankingEvaluator

/tmp/ipykernel_1092/3150954647.py:12: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import InformationRetrievalEvaluator


In [3]:
dataset = load_dataset("pameydorke/arcanumchat-cross-platform-with-vague-queries-qwen-36-a3b-fp8")

README.md:   0%|          | 0.00/622 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.20MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  118kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  120kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4600 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/574 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/577 [00:00<?, ? examples/s]

In [4]:
traind_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()
test_df = dataset['test'].to_pandas()
full_df = pd.concat([traind_df, val_df, test_df])

unique_docs_df = full_df.drop_duplicates(subset=['text'])
unique_docs_df = unique_docs_df.reset_index()
unique_docs_df = unique_docs_df.drop(columns=['query'])

unique_docs_ds = Dataset.from_pandas(unique_docs_df)

unique_docs_ds

Dataset({
    features: ['index', 'doc_id', 'link', 'title', 'text'],
    num_rows: 1151
})

In [11]:
DATA_URL = "https://raw.githubusercontent.com/ankorn/arcanumsearch/refs/heads/main/data/retriever_test_ds.json"
test_ds = requests.get(DATA_URL, timeout=30).json()

In [3]:
BASE_RETRIEVER_MODEL_ID = "BAAI/bge-m3"
FT_RETRIEVER_MODEL_ID = "pameydorke/arcanumchat-retriever-bge-m3"

BASE_RERANKER_MODEL_ID = "BAAI/bge-reranker-v2-m3"
FT_RERANKER_MODEL_ID = "pameydorke/arcanumchat-reranker-bge-reranker-v2-m3"

In [4]:
ft_reranker_model = CrossEncoder(FT_RERANKER_MODEL_ID, num_labels=1)

modules.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/262 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/35.2k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/225 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

In [7]:

base_retriever_model = SentenceTransformer(BASE_RETRIEVER_MODEL_ID)
ft_retriever_model = SentenceTransformer(FT_RETRIEVER_MODEL_ID)

base_reranker_model = CrossEncoder(BASE_RERANKER_MODEL_ID, num_labels=1)
ft_reranker_model = CrossEncoder(FT_RERANKER_MODEL_ID, num_labels=1)

base_retriever_model.eval()
ft_retriever_model.eval()
base_reranker_model.eval()
ft_reranker_model.eval()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

CrossEncoder(
  (0): Transformer({'transformer_task': 'sequence-classification', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'logits'}}, 'module_output_name': 'scores', 'architecture': 'XLMRobertaForSequenceClassification'})
)

In [8]:
queries = {}
corpus = {}
relevant_docs = {}
url2text = {}

unique_docs = {}
for idx, example in enumerate(unique_docs_ds):
    doc_link = example["link"]
    doc_text = example["text"]
    url2text[doc_link] = doc_text

    if doc_text not in unique_docs:
        doc_id = example["doc_id"]
        unique_docs[doc_text] = doc_id
        corpus[doc_id] = doc_text

skipped_counter = 0
vague_counter = 0
for idx, example in enumerate(test_ds):
    # if example['is_vague']:
    #     vague_counter += 1
    #     continue

    query = example['query']
    url = example['url']

    if 'https://www.reddit.com' in url:
        url = url.replace('https://www.reddit.com', 'https://reddit.com')

    if url not in url2text:
        skipped_counter += 1
        continue

    doc_text = url2text[url]

    if doc_text not in unique_docs:
        skipped_counter += 1
        continue

    doc_id = unique_docs[doc_text]

    query_id = f"query_{idx}"
    queries[query_id] = query
    relevant_docs[query_id] = {doc_id}

print("Skipped:", skipped_counter)
print("Skipped vague:", vague_counter)

Skipped: 2
Skipped vague: 0


In [ ]:
base_retriever_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="arcanum-retriever-test",
)
ft_retriever_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="arcanum-retriever-test",
)

base_retriever_results = base_retriever_evaluator(base_retriever_model)
ft_retriever_results = ft_retriever_evaluator(ft_retriever_model)

In [11]:
base_retriever_results

{'arcanum-retriever-test_cosine_accuracy@1': 0.6578947368421053,
 'arcanum-retriever-test_cosine_accuracy@3': 0.7842105263157895,
 'arcanum-retriever-test_cosine_accuracy@5': 0.8473684210526315,
 'arcanum-retriever-test_cosine_accuracy@10': 0.868421052631579,
 'arcanum-retriever-test_cosine_precision@1': 0.6578947368421053,
 'arcanum-retriever-test_cosine_precision@3': 0.26140350877192975,
 'arcanum-retriever-test_cosine_precision@5': 0.16947368421052625,
 'arcanum-retriever-test_cosine_precision@10': 0.08684210526315787,
 'arcanum-retriever-test_cosine_recall@1': 0.6578947368421053,
 'arcanum-retriever-test_cosine_recall@3': 0.7842105263157895,
 'arcanum-retriever-test_cosine_recall@5': 0.8473684210526315,
 'arcanum-retriever-test_cosine_recall@10': 0.868421052631579,
 'arcanum-retriever-test_cosine_ndcg@10': 0.7664273704408339,
 'arcanum-retriever-test_cosine_mrr@10': 0.7331766917293234,
 'arcanum-retriever-test_cosine_map@100': 0.7376623485506122}

In [12]:
ft_retriever_results

{'arcanum-retriever-test_cosine_accuracy@1': 0.6105263157894737,
 'arcanum-retriever-test_cosine_accuracy@3': 0.8,
 'arcanum-retriever-test_cosine_accuracy@5': 0.8736842105263158,
 'arcanum-retriever-test_cosine_accuracy@10': 0.9263157894736842,
 'arcanum-retriever-test_cosine_precision@1': 0.6105263157894737,
 'arcanum-retriever-test_cosine_precision@3': 0.2666666666666666,
 'arcanum-retriever-test_cosine_precision@5': 0.1747368421052631,
 'arcanum-retriever-test_cosine_precision@10': 0.09263157894736841,
 'arcanum-retriever-test_cosine_recall@1': 0.6105263157894737,
 'arcanum-retriever-test_cosine_recall@3': 0.8,
 'arcanum-retriever-test_cosine_recall@5': 0.8736842105263158,
 'arcanum-retriever-test_cosine_recall@10': 0.9263157894736842,
 'arcanum-retriever-test_cosine_ndcg@10': 0.7667290322679688,
 'arcanum-retriever-test_cosine_mrr@10': 0.7157602339181285,
 'arcanum-retriever-test_cosine_map@100': 0.7193342021850796}

In [ ]:
# base_reranker_evaluator = CrossEncoderRerankingEvaluator(
#     samples=hard_eval_samples,
#     at_k=10,
#     name="val-hard-negatives",
#     batch_size=64,
#     show_progress_bar=True,
#     always_rerank_positives=True,
# )
# ft_reranker_evaluator = CrossEncoderRerankingEvaluator(
#     samples=hard_eval_samples,
#     at_k=10,
#     name="val-hard-negatives",
#     batch_size=64,
#     show_progress_bar=True,
#     always_rerank_positives=True,
# )

# base_reranker_results = base_reranker_evaluator(base_reranker_model)
# ft_reranker_results = ft_reranker_evaluator(ft_reranker_model)

In [11]:
import math
import torch
import numpy as np
import pandas as pd
from sentence_transformers import util


def encode_queries_and_corpus(model, queries, corpus, batch_size=64):
    qids = list(queries.keys())
    cids = list(corpus.keys())

    q_texts = [queries[q] for q in qids]
    c_texts = [corpus[c] for c in cids]

    q_emb = model.encode(
        q_texts,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    c_emb = model.encode(
        c_texts,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return qids, cids, q_emb, c_emb

In [ ]:
import torch
import numpy as np
from sentence_transformers import util
import ir_measures
from ir_measures import nDCG, RR, Recall
import hashlib

def per_query_metrics(model, reranker, queries, corpus, relevant_docs, k_values=(1, 3, 5, 10), k_retrieval=10):
    assert k_retrieval >= max(k_values)
    assert not model.training, "bi-encoder in train mode"

    qids, cids, q_emb, c_emb = encode_queries_and_corpus(model, queries, corpus)

    sim = util.cos_sim(q_emb, c_emb)
    initial_ranked_indices = torch.argsort(sim, dim=1, descending=True)[:, :k_retrieval]
    initial_ranked_cids = [[cids[i] for i in row] for row in initial_ranked_indices.tolist()]

    reranked_cids = []
    for i, qid in enumerate(queries):
        query_text = queries[qid]
        candidate_cids = initial_ranked_cids[i]
        candidate_docs = [corpus[cid] for cid in candidate_cids]

        pairs = [(query_text, doc) for doc in candidate_docs]
        scores = reranker.predict(pairs, convert_to_numpy=True)

        assert scores.ndim == 1, f"predict returned {scores.shape}"

        order = np.argsort(scores)[::-1]
        reranked_cids.append([candidate_cids[j] for j in order])

    # Build qrels and run dictionaries for ir_measures
    qrels = {qid: {doc_id: 1 for doc_id in rel_set} for qid, rel_set in relevant_docs.items()}
    run = {
        qid: {doc_id: float(len(reranked_cids[i]) - rank) for rank, doc_id in enumerate(reranked_cids[i])}
        for i, qid in enumerate(qids)
    }

    measures = []
    for k in k_values:
        measures.extend([Recall @ k, nDCG @ k])
    measures.append(RR)  # RR gives reciprocal rank per query; averaging yields MRR

    results = {qid: {} for qid in qids}
    for metric in ir_measures.iter_calc(measures, qrels, run):
        qid = metric.query_id
        name = str(metric.measure)
        if name == "RR":
            name = "mrr"  # Per-query RR is the MRR contribution
        results[qid][name] = metric.value

    return results

In [ ]:
recall_metrics = ["R@3", "R@5", "R@10"]
ndcg_metrics = ["nDCG@10"]

def print_metrics(metrics, base_per_q, ft_per_q):
    for metric in metrics:
        base = np.array([base_per_q[q][metric] for q in queries])
        ft   = np.array([ft_per_q[q][metric]   for q in queries])
        delta = ft - base

        print("base mean:", base.mean())
        print("ft mean:  ", ft.mean())
        print("improved:", (delta > 0).sum())
        print("worse:   ", (delta < 0).sum())
        print("same:    ", (delta == 0).sum())

        rng = np.random.default_rng(0)
        n = len(delta)

        boot = np.array([
            delta[rng.integers(0, n, n)].mean()
            for _ in range(10_000)
        ])

        lo, hi = np.percentile(boot, [2.5, 97.5])
        print(f"mean delta = {delta.mean():.4f}")
        print(f"95% CI     = [{lo:.4f}, {hi:.4f}]", "\n")

In [ ]:
base_per_q = per_query_metrics(base_retriever_model, base_reranker_model, queries, corpus, relevant_docs, k_retrieval=10)

ft_per_q = per_query_metrics(ft_retriever_model, ft_reranker_model, queries, corpus, relevant_docs, k_retrieval=10)

In [ ]:
print_metrics(recall_metrics, base_per_q, ft_per_q)

base mean: 0.8105263157894737
ft mean:   0.868421052631579
improved: 18
worse:    7
same:     165
mean delta = 0.0579
95% CI     = [0.0053, 0.1105] 

base mean: 0.8526315789473684
ft mean:   0.9
improved: 17
worse:    8
same:     165
mean delta = 0.0474
95% CI     = [-0.0053, 0.1000] 

base mean: 0.868421052631579
ft mean:   0.9263157894736842
improved: 17
worse:    6
same:     167
mean delta = 0.0579
95% CI     = [0.0105, 0.1053] 



In [ ]:
print_metrics(ndcg_metrics, base_per_q, ft_per_q)

base mean: 0.7858541146451928
ft mean:   0.8540099243770441
improved: 35
worse:    13
same:     142
mean delta = 0.0682
95% CI     = [0.0248, 0.1117] 



In [ ]:
# base_ft_per_q = per_query_metrics(base_retriever_model, ft_reranker_model, queries, corpus, relevant_docs, k_retrieval=10)
# ft_base_per_q = per_query_metrics(ft_retriever_model,  base_reranker_model, queries, corpus, relevant_docs, k_retrieval=10)

In [26]:
base_per_q_20 = per_query_metrics(base_retriever_model, base_reranker_model, queries, corpus, relevant_docs, k_retrieval=20)
ft_per_q_20 = per_query_metrics(ft_retriever_model, ft_reranker_model, queries, corpus, relevant_docs, k_retrieval=20)

[diag] top10-pool md5=90fcc4a1879dc899
[diag] top10-pool md5=42705545208c6318


In [27]:
print_metrics(recall_metrics, base_per_q_20, ft_per_q_20)

base mean: 0.8157894736842105
ft mean:   0.9
improved: 18
worse:    2
same:     170
mean delta = 0.0842
95% CI     = [0.0421, 0.1316] 

base mean: 0.8631578947368421
ft mean:   0.9263157894736842
improved: 15
worse:    3
same:     172
mean delta = 0.0632
95% CI     = [0.0211, 0.1053] 

base mean: 0.8947368421052632
ft mean:   0.9526315789473684
improved: 13
worse:    2
same:     175
mean delta = 0.0579
95% CI     = [0.0211, 0.1000] 



In [28]:
print_metrics(ndcg_metrics, base_per_q_20, ft_per_q_20)

base mean: 0.7907573475907835
ft mean:   0.8724546898622102
improved: 39
worse:    11
same:     140
mean delta = 0.0817
95% CI     = [0.0484, 0.1175] 



In [29]:
rows = []
for qid in queries:
    row = {"qid": qid, "query": queries[qid]}
    for k, v in base_per_q[qid].items():
        row[f"base_{k}"] = v
    for k, v in ft_per_q[qid].items():
        row[f"ft_{k}"] = v
    rows.append(row)

df = pd.DataFrame(rows)
df["delta_ndcg@10"] = df["ft_nDCG@10"] - df["base_nDCG@10"]
df = df.sort_values("delta_ndcg@10")

# Queries where fine-tuning hurt most
print(df[["qid", "query", "base_nDCG@10", "ft_nDCG@10", "delta_ndcg@10"]].head(20))

           qid                                   query  base_nDCG@10  \
23    query_23                              find elves      1.000000   
61    query_63                         ancesctral swrd      1.000000   
105  query_107                                   dogde      1.000000   
10    query_10        Arcane Sword dungeon coordinates      0.630930   
135  query_137                    sniper rifgle schema      0.630930   
69    query_71                               haf ogres      1.000000   
36    query_36                                     UAP      0.500000   
99   query_101  Tarnat guards attack after killing npc      1.000000   
179  query_181                Eshburry throwing expert      1.000000   
15    query_15                  bow mastr trainee find      0.630930   
19    query_19                         orcs sttus docs      0.430677   
21    query_21     steem community patch does not work      0.386853   
96    query_98                         gambling master      0.43

In [30]:
# Build a list of (qid, base, ft, delta) for recall@5
rows = []
for qid in queries:
    b = base_per_q[qid]["R@10"]
    f = ft_per_q[qid]["R@10"]
    rows.append((qid, b, f, f - b))

# Keep only the ones that got worse
worse = [r for r in rows if r[3] < 0]
worse.sort(key=lambda r: r[3])  # most negative first

print(f"Total worse on recall@10: {len(worse)}")
print("=" * 80)

for qid, b, f, d in worse:
    # print(f"qid:     {qid}")
    print(f"query:   {queries[qid]}")
    # print(f"base:    {b:.3f}   ft: {f:.3f}   delta: {d:+.3f}")

    rel_docs = relevant_docs[qid]
    # print(f"relevant doc ids: {rel_docs}")

    # Show the text of the relevant docs (truncated)
    for doc_id in rel_docs:
        text = corpus[doc_id]
        # print(f"  [{doc_id}] {text[:300]}...")

    # If you want to see where the fine-tuned model ranked the gold doc,
    # you need the ranked list. See helper below.
    print("-" * 80)

Total worse on recall@10: 6
query:   Arcane Sword dungeon coordinates
--------------------------------------------------------------------------------
query:   find elves
--------------------------------------------------------------------------------
query:   UAP
--------------------------------------------------------------------------------
query:   ancesctral swrd
--------------------------------------------------------------------------------
query:   dogde
--------------------------------------------------------------------------------
query:   sniper rifgle schema
--------------------------------------------------------------------------------
